# TavilyCrawl 教程:智能网页爬取

## 我们将构建什么

在本教程中,你将学习如何使用 TavilyCrawl 通过 AI 引导指令智能地爬取网站。我们将演示:

1. **基础网页爬取** - 不带特定指令爬取一个网站
2. **指令引导爬取** - 使用自然语言定位特定内容
3. **结果对比** - 对比两种方式的效果
4. **最佳实践** - 学习如何编写高效的爬取指令

### 目标网站
我们将爬取 LangChain 文档 (https://python.langchain.com/) 来查找关于 AI Agent 的内容。

## 什么是 TavilyCrawl?

TavilyCrawl 是一款智能网页爬虫,它使用 AI 来决定爬取过程中要探索哪些路径。它将 AI 驱动的决策与并行处理能力相结合。

### 核心特性:

- **AI 驱动路径选择**:使用 AI 决定要探索哪些路径
- **并行处理**:同时探索数百条路径
- **高级内容提取**:从动态渲染的页面中提取内容
- **指令驱动**:遵循自然语言指令来引导探索
- **定向内容**:返回为 LLM 集成和 RAG 系统定制的内容

### Tavily 资源:
- <a href="https://tavily.com" target="_blank">官方网站</a>
- <a href="https://docs.tavily.com" target="_blank">API 文档</a>
- <a href="https://docs.tavily.com/documentation/api-reference/endpoint/crawl" target="_blank">Crawl API 参考</a>
- <a href="https://pypi.org/project/langchain-tavily/" target="_blank">LangChain Python 集成</a>
- <a href="https://app.tavily.com/home?utm_campaign=eden_marco&utm_medium=socials&utm_source=linkedin" target="_blank">获取 API Key</a>

本教程通过在 LangChain 文档上对比带指令和不带指令的爬取结果来演示 TavilyCrawl。

---

## 安装与环境配置

首先,让我们安装所需的包并配置环境。

In [ ]:
# 安装所需包
%pip install langchain-tavily certifi

# 用于美化打印和可视化
%pip install rich pandas

In [ ]:
import os
import ssl
import json
from typing import Any, Dict, List

import certifi
from langchain_tavily import TavilyCrawl
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.json import JSON

# 配置 SSL 上下文(解决 Windows 证书验证问题)
ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# 初始化 rich console 用于美化打印
console = Console()

print("所有导入成功!")

## API Key 配置

你需要一个 Tavily API key 才能使用 TavilyCrawl。前往 [https://app.tavily.com/home?utm_campaign=eden_marco&utm_medium=socials&utm_source=linkedin](https://app.tavily.com/home?utm_campaign=eden_marco&utm_medium=socials&utm_source=linkedin) 获取。

设置环境变量 `TAVILY_API_KEY`

In [ ]:
# 在此设置你的 Tavily API key
import getpass

# 对于 Google Colab,可以使用 getpass 进行安全输入
if 'TAVILY_API_KEY' not in os.environ:
    os.environ['TAVILY_API_KEY'] = getpass.getpass('输入你的 Tavily API key: ')

# 备选:直接设置(取消注释并填入你的 key)
# os.environ["TAVILY_API_KEY"] = "your_tavily_api_key_here"

print("API key 设置成功!")

## 初始化 TavilyCrawl

初始化 TavilyCrawl 并设置演示用的目标 URL。

In [ ]:
# 初始化 TavilyCrawl
tavily_crawl = TavilyCrawl()

# 目标 URL:LangChain 文档
target_url = "https://python.langchain.com/"

console.print(Panel.fit(
    f"目标网站: {target_url}\n爬虫: TavilyCrawl",
    title="演示设置",
    border_style="bright_blue"
))

print("TavilyCrawl 初始化成功")

## 演示 1:不带指令爬取

不带特定指令进行爬取,以展示在 LangChain 文档上的基准行为。

In [ ]:
# 演示 1:不带指令爬取
console.print(Panel.fit(
    f"目标: {target_url}\n指令: 无(基准爬取)\n最大深度: 1\n提取深度: advanced",
    title="演示 1:不带指令爬取",
    border_style="blue"
))

console.print("正在运行不带指令的 TavilyCrawl...", style="blue")

# 不带指令的基础爬取
basic_result = tavily_crawl.invoke({
    "url": target_url,
    "max_depth": 1,
    "extract_depth": "advanced"
})

# 立即显示原始输出
console.print(basic_result)

# 提取结果用于分析
basic_results = basic_result.get("results", [])

# 现在以美观的格式展示结果

In [ ]:
console.print(f"\n不带指令的结果: {len(basic_results)} 个页面", style="cyan")
console.print("   包含 LangChain 文档中所有类型的内容")
console.print("   未经过滤 - 爬取区域内的所有内容")
console.print("   需要手动操作才能找到特定内容")

console.print("\n基础爬取(未过滤)的示例结果:\n", style="cyan")

for i, result in enumerate(basic_results[:3], 1):  # 显示前 3 条结果
    url = result.get("url", "无 URL")
    content = result.get("raw_content", "无内容")[:150] + "..."
    
    panel_content = f"""URL: {url}

内容预览:
{content}"""
    
    console.print(Panel(
        panel_content,
        title=f"{i}. {url}",
        border_style="blue"
    ))
    print()

console.print(f"... 以及其他 {len(basic_results) - 3} 条混合结果", style="italic cyan")
console.print("注意:内容类型混杂 - 指南、集成、概念等。", style="cyan")

## 演示 2:带指令爬取

使用特定指令来提高爬取结果的质量和相关性。指令可以显著改善目标定位和过滤效果。

In [ ]:
instructions = "找到所有关于 ai agents 的页面"

console.print(Panel.fit(
    f"目标: {target_url} (与演示 1 相同)\n指令: {instructions}\n类型: 具体、行动导向\n最大深度: 3\n提取深度: advanced",
    title="演示 2:带指令爬取",
    border_style="green"
))

console.print("开始带指令的爬取...", style="green")
console.print("指令将引导 AI 定位特定内容", style="italic")

In [ ]:
# 执行带指令的爬取
result_with_instructions = tavily_crawl.invoke({
    "url": target_url,
    "instructions": instructions,
    "max_depth": 3,
    "extract_depth": "advanced"
})

# 立即显示原始输出
console.print("\nTavilyCrawl 原始输出:", style="yellow")
console.print(result_with_instructions)

console.print("\n带指令的爬取已完成", style="green")

# 展示基于指令过滤的结果
results_with_instructions = result_with_instructions.get("results", [])

In [ ]:
# 展示找到的定向 Agent 文档
console.print("\n找到的 LangChain Agent 文档:\n", style="green")

for i, result in enumerate(results_with_instructions, 1):
    url = result.get("url", "无 URL")
    content = result.get("raw_content", "无内容")[:200] + "..."
    
    panel_content = f"""URL: {url}

内容预览:
{content}"""
    
    console.print(Panel(
        panel_content,
        title=f"{i}. {url}",
        border_style="green"
    ))
    print()

console.print("注意:所有结果都是关于 LangChain 中 Agent 的内容", style="green")

## 两种方式对比

对比两种方式,理解指令质量的影响。

In [ ]:
# 创建对比表格
comparison_table = Table(title="TavilyCrawl:指令质量对比")
comparison_table.add_column("方式", style="cyan", no_wrap=True)
comparison_table.add_column("指令", style="yellow")
comparison_table.add_column("找到的页面数", style="blue")
comparison_table.add_column("内容质量", style="green")
comparison_table.add_column("实用性", style="red")

comparison_table.add_row(
    "无指令",
    "无(基准)",
    f"{len(basic_results)}",
    "混杂(所有类型)",
    "低(需要过滤)"
)

comparison_table.add_row(
    "带指令",
    instructions,
    f"{len(results_with_instructions)}",
    "高度定向",
    "高(可直接使用)"
)

console.print(comparison_table)

console.print("\n关键观察:", style="blue")
console.print("   无指令会返回所有内容,需要手动过滤")
console.print("   指令提供高度定向、可直接使用的结果")
console.print("   最佳实践:使用具体、行动导向的指令")

console.print(f"\n使用指令的效率:", style="green")
console.print(f"   过滤效率: 噪音降低 {((len(basic_results) - len(results_with_instructions)) / len(basic_results) * 100):.1f}%")
console.print("   节省时间: 无需手动后处理")
console.print("   人工智能驱动: 智能路径选择与内容过滤")